In [1]:
#| export

# Essential imports for data loading
import random
import sys
import time
from abc import ABC, abstractmethod
from typing import Iterator, List, Tuple

import numpy as np

# Import real Tensor class from tinytorch package
from tinytorch.core.tensor import Tensor

In [2]:
#| export
class Dataset(ABC):
    """
    Abstract base class for all datasets.

    Provides the fundamental interface that all datasets must implement:
    - __len__(): Returns the total number of samples
    - __getitem__(idx): Returns the sample at given index

    TODO: Implement the abstract Dataset base class

    APPROACH:
    1. Use ABC (Abstract Base Class) to define interface
    2. Mark methods as @abstractmethod to force implementation
    3. Provide clear docstrings for subclasses

    EXAMPLE:
    >>> class MyDataset(Dataset):
    ...     def __len__(self): return 100
    ...     def __getitem__(self, idx): return idx
    >>> dataset = MyDataset()
    >>> print(len(dataset))  # 100
    >>> print(dataset[42])   # 42

    HINT: Abstract methods force subclasses to implement core functionality
    """

    ### BEGIN SOLUTION
    @abstractmethod
    def __len__(self) -> int:
        """
        Return the total number of samples in the dataset.

        This method must be implemented by all subclasses to enable
        len(dataset) calls and batch size calculations.
        """
        pass

    @abstractmethod
    def __getitem__(self, idx: int):
        """
        Return the sample at the given index.

        Args:
            idx: Index of the sample to retrieve (0 <= idx < len(dataset))

        Returns:
            The sample at index idx. Format depends on the dataset implementation.
            Could be (data, label) tuple, single tensor, etc.
        """
        pass



In [5]:
def test_unit_dataset():
    """🧪 Test Dataset abstract base class."""
    print("🧪 Unit Test: Dataset Abstract Base Class...")

    # Test that Dataset is properly abstract
    try:
        dataset = Dataset()
        assert False, "Should not be able to instantiate abstract Dataset"
    except TypeError:
        print("✅ Dataset is properly abstract")

    # Test concrete implementation
    class TestDataset(Dataset):
        def __init__(self, size):
            self.size = size

        def __len__(self):
            # print(self.size)
            return self.size

        def __getitem__(self, idx):
            # print(self.size)

            return f"item_{idx}"

    dataset = TestDataset(10)
    assert len(dataset) == 10
    assert dataset[0] == "item_0"
    assert dataset[9] == "item_9"

    print("✅ Dataset interface works correctly!")

if __name__ == "__main__":
    test_unit_dataset()

🧪 Unit Test: Dataset Abstract Base Class...
✅ Dataset is properly abstract
✅ Dataset interface works correctly!


In [33]:
#| export
class TensorDataset(Dataset):
    """
    Dataset wrapping tensors for supervised learning.

    Each sample is a tuple of tensors from the same index across all input tensors.
    All tensors must have the same size in their first dimension.

    TODO: Implement TensorDataset for tensor-based data

    APPROACH:
    1. Store all input tensors
    2. Validate they have same first dimension (number of samples)
    3. Return tuple of tensor slices for each index

    EXAMPLE:
    >>> features = Tensor([[1, 2], [3, 4], [5, 6]])  # 3 samples, 2 features each
    >>> labels = Tensor([0, 1, 0])                    # 3 labels
    >>> dataset = TensorDataset(features, labels)
    >>> print(len(dataset))  # 3
    >>> print(dataset[1])    # (Tensor([3, 4]), Tensor(1))

    HINTS:
    - Use *tensors to accept variable number of tensor arguments
    - Check all tensors have same length in dimension 0
    - Return tuple of tensor[idx] for all tensors
    """

    def __init__(self, *tensors):
        """
        Create dataset from multiple tensors.

        Args:
            *tensors: Variable number of Tensor objects

        All tensors must have the same size in their first dimension.
        """
        ### BEGIN SOLUTION
        assert len(tensors) > 0, "Must provide at least one tensor"

        self.tensors = tensors
        first_size = len(tensors[0].data)
        # print(tensors)
        # print(tensors[0])
        # print(tensors[0].data)
        # print(first_size)

        for i, tensor in enumerate(tensors):
            if len(tensor.data) != first_size:
                raise ValueError(
                    f"Tensor size mismatch in TensorDataset\n"
                    f"  ❌ Tensor 0 has {first_size} samples, but Tensor {i} has {len(tensor.data)} samples\n"
                    f"  💡 All tensors must have the same size in their first dimension (the sample dimension)\n"
                    f"  🔧 Check your data: features.shape[0] should equal labels.shape[0]\n"
                    f"     Example fix: labels = labels[:{first_size}] or features = features[:{len(tensor.data)}]"
                )
            


    def __len__(self) -> int:
        """
        Return number of samples (size of first dimension).

        TODO: Return the total number of samples in the dataset

        APPROACH:
        1. Access the first tensor from self.tensors
        2. Return length of its data (first dimension size)

        EXAMPLE:
        >>> features = Tensor([[1, 2], [3, 4], [5, 6]])  # 3 samples
        >>> labels = Tensor([0, 1, 0])
        >>> dataset = TensorDataset(features, labels)
        >>> print(len(dataset))  # 3

        HINT: All tensors have same first dimension (validated in __init__)
        """
        # pass
        return len(self.tensors[0].data)

    def __getitem__(self, idx: int) -> Tuple[Tensor]:
        """
        Return tuple of tensor slices at given index.

        TODO: Return the sample at the given index

        APPROACH:
        1. Validate index is within bounds
        2. Extract data at index from each tensor
        3. Wrap each slice in a Tensor and return as tuple

        Args:
            idx: Sample index

        Returns:
            Tuple containing tensor[idx] for each input tensor

        EXAMPLE:
        >>> features = Tensor([[1, 2], [3, 4], [5, 6]])
        >>> labels = Tensor([0, 1, 0])
        >>> dataset = TensorDataset(features, labels)
        >>> sample = dataset[1]
        >>> # Returns: (Tensor([3, 4]), Tensor(1))

        HINTS:
        - Check idx < len(self) to prevent out-of-bounds access
        - Use generator expression with tuple() for clean syntax
        """
        # pass
        if idx >= len(self) or idx < 0:
            raise IndexError(f"Index {idx} out of range for dataset of size {len(self)}")

        # for tensor in self.tensors:
        #     print(tensor.data[idx])
        return tuple(Tensor(tensor.data[idx]) for tensor in self.tensors) # loop all the data, labels


In [31]:
features = Tensor([[1, 2], [3, 4]])
labels = Tensor([0, 1])
x = TensorDataset(features, labels)

sample = x[1]
print(sample)

[3. 4.]
1.0
(Tensor(data=[3. 4.], shape=(2,)), Tensor(data=1.0, shape=()))


In [34]:
def test_unit_tensordataset():
    """🧪 Test TensorDataset implementation."""
    print("🧪 Unit Test: TensorDataset...")

    # Test basic functionality
    features = Tensor([[1, 2], [3, 4], [5, 6]])  # 3 samples, 2 features
    labels = Tensor([0, 1, 0])                   # 3 labels

    dataset = TensorDataset(features, labels)

    # Test length
    assert len(dataset) == 3, f"Expected length 3, got {len(dataset)}"

    # Test indexing
    sample = dataset[0]
    assert len(sample) == 2, "Should return tuple with 2 tensors"
    assert np.array_equal(sample[0].data, [1, 2]), f"Wrong features: {sample[0].data}"
    assert sample[1].data == 0, f"Wrong label: {sample[1].data}"

    sample = dataset[1]
    assert np.array_equal(sample[1].data, 1), f"Wrong label at index 1: {sample[1].data}"

    # Test error handling
    try:
        dataset[10]  # Out of bounds
        assert False, "Should raise IndexError for out of bounds access"
    except IndexError:
        pass

    # Test mismatched tensor sizes
    try:
        bad_features = Tensor([[1, 2], [3, 4]])  # Only 2 samples
        bad_labels = Tensor([0, 1, 0])           # 3 labels - mismatch!
        TensorDataset(bad_features, bad_labels)
        assert False, "Should raise error for mismatched tensor sizes"
    except ValueError:
        pass

    print("✅ TensorDataset works correctly!")

if __name__ == "__main__":
    test_unit_tensordataset()

🧪 Unit Test: TensorDataset...
✅ TensorDataset works correctly!


In [37]:
#| export
class DataLoader:
    """
    Data loader with batching and shuffling support.

    Wraps a dataset to provide batched iteration with optional shuffling.
    Essential for efficient training with mini-batch gradient descent.

    TODO: Implement DataLoader with batching and shuffling

    APPROACH:
    1. Store dataset, batch_size, and shuffle settings
    2. Create iterator that groups samples into batches
    3. Handle shuffling by randomizing indices
    4. Collate individual samples into batch tensors

    EXAMPLE:
    >>> dataset = TensorDataset(Tensor([[1,2], [3,4], [5,6]]), Tensor([0,1,0]))
    >>> loader = DataLoader(dataset, batch_size=2, shuffle=True)
    >>> for batch in loader:
    ...     features_batch, labels_batch = batch
    ...     print(f"Features: {features_batch.shape}, Labels: {labels_batch.shape}")

    HINTS:
    - Use random.shuffle() for index shuffling
    - Group consecutive samples into batches
    - Stack individual tensors using np.stack()
    """

    def __init__(self, dataset: Dataset, batch_size: int, shuffle: bool = False):
        """
        Create DataLoader for batched iteration.

        Args:
            dataset: Dataset to load from
            batch_size: Number of samples per batch
            shuffle: Whether to shuffle data each epoch
        """
        self.dataset = dataset
        self.batch_size = batch_size
        self.shuffle = shuffle
    
    def __len__(self) -> int:
        """
        Return number of batches per epoch.

        TODO: Calculate the number of batches based on dataset size and batch_size

        APPROACH:
        1. Use ceiling division: (dataset_size + batch_size - 1) // batch_size
        2. This ensures we count the last partial batch

        EXAMPLE:
        >>> dataset = TensorDataset(Tensor([[1], [2], [3], [4], [5]]))
        >>> loader = DataLoader(dataset, batch_size=2)
        >>> print(len(loader))  # 3 (batches: [2, 2, 1])

        HINT: Ceiling division handles uneven splits correctly
        """
        return (len(self.dataset) + self.batch_size - 1) // self.batch_size

    def __iter__(self) -> Iterator:
        """
        Return iterator over batches.

        TODO: Implement iteration that yields batches of data

        APPROACH:
        1. Create list of indices [0, 1, 2, ..., len(dataset)-1]
        2. Shuffle indices if self.shuffle is True
        3. Group indices into chunks of batch_size
        4. For each chunk, retrieve samples and collate into batch

        EXAMPLE:
        >>> dataset = TensorDataset(Tensor([[1], [2], [3], [4]]))
        >>> loader = DataLoader(dataset, batch_size=2)
        >>> for batch in loader:
        ...     print(batch[0].shape)  # (2, 1)

        HINTS:
        - Use random.shuffle() to randomize indices
        - Use range(0, len(indices), batch_size) to create chunks
        - Call self._collate_batch() to convert list of samples to batch tensors
        """
        indices = list(range(len(self.dataset)))
        if self.shuffle:
            random.shuffle(indices)

        # yielf batches
        for i in range(0, len(indices), self.batch_size):
            batch_indices = indices[i:i + self.batch_size]
            batch = [self.dataset[idx] for idx in batch_indices]

            yield self._collate_batch(batch)

    def _collate_batch(self, batch: List[Tuple[Tensor, ...]]) -> Tuple[Tensor, ...]:
        """
        Collate individual samples into batch tensors.

        TODO: Stack individual sample tensors into batch tensors

        APPROACH:
        1. Handle empty batch edge case
        2. Determine how many tensors per sample (e.g., 2 for features + labels)
        3. For each tensor position, extract all samples at that position
        4. Stack them using np.stack() to create batch dimension
        5. Wrap result in Tensor and return tuple

        Args:
            batch: List of sample tuples from dataset

        Returns:
            Tuple of batched tensors

        EXAMPLE:
        >>> # batch = [(Tensor([1,2]), Tensor(0)),
        ...            (Tensor([3,4]), Tensor(1))]
        >>> # Returns: (Tensor([[1,2], [3,4]]), Tensor([0, 1]))

        HINTS:
        - Use len(batch[0]) to get number of tensors per sample
        - Extract .data from each tensor before stacking
        - np.stack() creates new axis at position 0 (batch dimension)
        """

        if len(batch) == 0:
            return ()
        
        # determine number of tensors per sample
        num_tensors = len(batch[0])

        batched_tensors = []
        for tensor_idx in range(num_tensors): # 0 = features, 1 = labels
            # extract all tensors at this position
            tensor_list = [sample[tensor_idx].data for sample in batch]

            batched_data = np.stack(tensor_list, axis=0)
            batched_tensors.append(Tensor(batched_data))

        return tuple(batched_tensors)




In [38]:
def get_batches(data, batch_size):
    for i in range(0, len(data), batch_size):
        print(f"  [generator] producing batch {i}")
        yield data[i:i+batch_size]
        print(f"  [generator] resumed, moving to next batch")

data = [1, 2, 3, 4, 5, 6]

for batch in get_batches(data, batch_size=2):
    print(f"[main] got batch: {batch}")
    print(f"[main] doing work on batch...")

  [generator] producing batch 0
[main] got batch: [1, 2]
[main] doing work on batch...
  [generator] resumed, moving to next batch
  [generator] producing batch 2
[main] got batch: [3, 4]
[main] doing work on batch...
  [generator] resumed, moving to next batch
  [generator] producing batch 4
[main] got batch: [5, 6]
[main] doing work on batch...
  [generator] resumed, moving to next batch


In [39]:
#| export

class RandomHorizontalFlip:
    """
    Randomly flip images horizontally with given probability.

    A simple but effective augmentation for most image datasets.
    Flipping is appropriate when horizontal orientation doesn't change class
    (cats, dogs, cars - not digits or text!).

    Args:
        p: Probability of flipping (default: 0.5)
    """

    def __init__(self, p=0.5):
        """
        Initialize RandomHorizontalFlip.

        TODO: Store flip probability

        APPROACH:
        1. Validate probability is in range [0, 1]
        2. Store p as instance variable

        EXAMPLE:
        >>> flip = RandomHorizontalFlip(p=0.5)  # 50% chance to flip

        HINT: Raise ValueError if p is outside valid range
        """
        if not 0.0 <= p <= 1.0:
            raise ValueError(
                f"Invalid flip probability: {p}\n"
                f"  ❌ p must be between 0.0 and 1.0\n"
                f"  💡 p is the probability of flipping the image horizontally (p=0.5 means 50% chance)\n"
                f"  🔧 Common values: p=0.0 (never flip), p=0.5 (standard), p=1.0 (always flip)"
            )
        self.p = p

    def __call__(self, x):
        """
        Apply random horizontal flip to input.

        TODO: Implement random horizontal flip

        APPROACH:
        1. Generate random number in [0, 1)
        2. If random < p, flip horizontally
        3. Otherwise, return unchanged

        Args:
            x: Input array with shape (..., H, W) or (..., H, W, C)
               Flips along the last-1 axis (width dimension)

        Returns:
            Flipped or unchanged array (same shape as input)

        EXAMPLE:
        >>> flip = RandomHorizontalFlip(0.5)
        >>> img = np.array([[1, 2, 3], [4, 5, 6]])  # 2x3 image
        >>> # 50% chance output is [[3, 2, 1], [6, 5, 4]]

        HINT: Think about all the possible position of the width axis to flip
        """

        if np.random.random() < self.p:
            is_tensor = isinstance(x, Tensor)
            data = x.data if is_tensor else x

            if data.ndim == 2:
                # (H, W)
                axis = -1
            elif data.ndim >= 3:
                if data.shape[-1] <= 4:
                    # Channels-last: (..., H, W, C)
                    axis = -2
                elif data.shape[-3] <= 4:
                    # Channels-first: (..., C, H, W)
                    axis = -1
                else:
                    axis = -1

            else:
                raise ValueError(
                    f"RandomHorizontalFlip requires at least 2D input\n"
                    f"  ❌ Got {data.ndim}D input with shape {data.shape}\n"
                    f"  💡 Images need at least height and width dimensions (H, W) to flip horizontally\n"
                    f"  🔧 Reshape your data: x.reshape(height, width) or x.reshape(1, height, width)"
                )

            flipped = np.flip(data, axis=axis).copy()
            return Tensor(flipped) if is_tensor else flipped
        return x

In [40]:
#| export
def _pad_image(data, padding):
    """
    Detect image format and apply zero-padding to spatial dimensions only.

    TODO: Pad the image with zeros on all spatial sides

    APPROACH:
    1. Check dimensionality: 2D (H,W), 3D (C,H,W or H,W,C), else error
    2. For 2D: pad uniformly on all sides
    3. For 3D: determine channel axis (first dim <= 4 → channels-first)
    4. Pad only H and W dimensions, leave channels untouched

    EXAMPLE:
    >>> img = np.ones((3, 4, 4))  # (C, H, W) = (3, 4, 4)
    >>> padded = _pad_image(img, padding=2)
    >>> print(padded.shape)  # (3, 8, 8) — channels unchanged, spatial +4

    HINTS:
    - np.pad takes a pad_width per axis: ((before, after), (before, after), ...)
    - Use (0, 0) for axes you do NOT want to pad (channels)
    - Use (padding, padding) for axes you DO want to pad (H, W)
    """

    if data.ndim == 2:
        # (H, W) format - pad both axes
        return np.pad(data, padding, mode="constant", constant_values=0)
    elif data.ndim == 3:
        if data.shape[0] <= 4:
            # (C, H, W) - pad only H and W
            return np.pad(data,
                          ((0,0), (padding,padding), (padding,padding)),
                          mode="constant", constant_values=0)
        else:
            # (H, W, C) - pad only H and W
            return np.pad(data,
                          ((padding,padding), (padding,padding),(0,0)),
                          mode="constant", constant_values=0)
        
    else:
        raise ValueError(
            f"RandomCrop requires 2D or 3D input\n"
            f"  ❌ Got {data.ndim}D input with shape {data.shape}\n"
            f"  💡 Expected formats: (H, W) for grayscale, (C, H, W) or (H, W, C) for color images\n"
            f"  🔧 Reshape your data:\n"
            f"     - For single grayscale image: x.reshape(height, width)\n"
            f"     - For single color image: x.reshape(channels, height, width)"
        )


In [41]:
def test_unit_pad_image():
    """🧪 Test _pad_image helper."""
    print("🧪 Unit Test: _pad_image...")

    # Test 2D (H, W) — grayscale
    img_hw = np.ones((8, 8))
    padded_hw = _pad_image(img_hw, padding=4)
    assert padded_hw.shape == (16, 16), f"2D pad shape wrong: {padded_hw.shape}"
    # Corners should be zero (padding), center should be ones
    assert padded_hw[0, 0] == 0.0, "Padding should be zero"
    assert padded_hw[4, 4] == 1.0, "Original data should be preserved"

    # Test 3D channels-first (C, H, W)
    img_chw = np.ones((3, 8, 8))
    padded_chw = _pad_image(img_chw, padding=4)
    assert padded_chw.shape == (3, 16, 16), f"CHW pad shape wrong: {padded_chw.shape}"
    assert padded_chw[0, 0, 0] == 0.0, "Spatial padding should be zero"
    assert padded_chw[0, 4, 4] == 1.0, "Original data should be preserved"

    # Test 3D channels-last (H, W, C) — shape[0] > 4 triggers this path
    img_hwc = np.ones((32, 32, 3))
    padded_hwc = _pad_image(img_hwc, padding=4)
    assert padded_hwc.shape == (40, 40, 3), f"HWC pad shape wrong: {padded_hwc.shape}"

    # Test error on 1D input
    try:
        _pad_image(np.ones((10,)), padding=4)
        assert False, "Should raise ValueError for 1D input"
    except ValueError:
        pass

    print("✅ _pad_image works correctly!")

if __name__ == "__main__":
    test_unit_pad_image()

🧪 Unit Test: _pad_image...
✅ _pad_image works correctly!


In [42]:
#| export
def _random_crop_region(padded_h, padded_w, target_h, target_w):
    """
    Sample a random (top, left) position for cropping.

    TODO: Generate random top-left corner within valid bounds

    APPROACH:
    1. Compute max valid top: padded_h - target_h
    2. Compute max valid left: padded_w - target_w
    3. Sample uniformly from [0, max_top] and [0, max_left]

    EXAMPLE:
    >>> top, left = _random_crop_region(40, 40, 32, 32)
    >>> # top in [0, 8], left in [0, 8]

    HINT: np.random.randint(0, high+1) gives values in [0, high] inclusive
    """

    top = np.random.randint(0, padded_h - target_h + 1)
    left = np.random.randint(0, padded_w - target_w + 1)
    return top, left


In [43]:
def test_unit_random_crop_region():
    """🧪 Test _random_crop_region helper."""
    print("🧪 Unit Test: _random_crop_region...")

    padded_h, padded_w = 40, 40
    target_h, target_w = 32, 32
    max_top = padded_h - target_h   # 8
    max_left = padded_w - target_w  # 8

    # Run many trials and verify bounds
    for _ in range(200):
        top, left = _random_crop_region(padded_h, padded_w, target_h, target_w)
        assert 0 <= top <= max_top, f"top={top} out of range [0, {max_top}]"
        assert 0 <= left <= max_left, f"left={left} out of range [0, {max_left}]"

    # Verify both endpoints are reachable (statistical test)
    tops = set()
    lefts = set()
    for _ in range(500):
        t, l = _random_crop_region(padded_h, padded_w, target_h, target_w)
        tops.add(t)
        lefts.add(l)

    assert 0 in tops, "Position 0 should be reachable"
    assert max_top in tops, f"Position {max_top} should be reachable"
    assert 0 in lefts, "Position 0 should be reachable"
    assert max_left in lefts, f"Position {max_left} should be reachable"

    # Edge case: when padded == target, only one valid position
    top, left = _random_crop_region(32, 32, 32, 32)
    assert top == 0 and left == 0, "Only valid position when padded == target"

    print("✅ _random_crop_region works correctly!")

if __name__ == "__main__":
    test_unit_random_crop_region()

🧪 Unit Test: _random_crop_region...
✅ _random_crop_region works correctly!


In [49]:
#| export
class RandomCrop:
    """
    Randomly crop image after padding.

    This is the standard augmentation for CIFAR-10:
    1. Pad image by `padding` pixels on each side
    2. Randomly crop back to original size

    This simulates small translations in the image, forcing the model
    to recognize objects regardless of their exact position.

    Args:
        size: Output crop size (int for square, or tuple (H, W))
        padding: Pixels to pad on each side before cropping (default: 4)
    """

    def __init__(self, size, padding=4):
        """
        Initialize RandomCrop.

        TODO: Store crop parameters

        APPROACH:
        1. Convert size to tuple if it's an int (for square crops)
        2. Store size and padding as instance variables

        EXAMPLE:
        >>> crop = RandomCrop(32, padding=4)  # CIFAR-10 standard
        >>> # Pads to 40x40, then crops back to 32x32

        HINT: Handle both int and tuple sizes for flexibility
        """

        if isinstance(size, int):
            self.size = (size, size)
        else:
            self.size = size
        self.padding = padding

    def __call__(self, x):
        """
        Apply random crop after padding.

        Composes _pad_image and _random_crop_region to perform:
        1. Pad the image with zeros on spatial dimensions
        2. Sample a random crop position
        3. Extract the crop and return

        TODO: Compose _pad_image → _random_crop_region → slice extraction

        APPROACH:
        1. Unwrap Tensor if needed, get raw numpy array
        2. Call _pad_image(data, self.padding) to pad spatial dims
        3. Determine padded spatial dimensions (H, W) based on format
        4. Call _random_crop_region(padded_h, padded_w, target_h, target_w)
        5. Slice the padded array at (top, left) for target size
        6. Re-wrap as Tensor if input was Tensor

        EXAMPLE:
        >>> crop = RandomCrop(32, padding=4)
        >>> img = np.random.randn(3, 32, 32)
        >>> out = crop(img)
        >>> print(out.shape)  # (3, 32, 32)

        HINT: The slicing pattern differs by format:
          2D:  padded[top:top+h, left:left+w]
          CHW: padded[:, top:top+h, left:left+w]
          HWC: padded[top:top+h, left:left+w, :]
        """

        is_tensor = isinstance(x, Tensor)
        data = x.data if is_tensor else x

        target_h, target_w = self.size

        # step 1: pad the image (handles format detection internally)
        padded = _pad_image(data, self.padding)

        # step 2: determine padded spatial dims and sample crop position
        if data.ndim == 2:
            padded_h, padded_w = padded.shape
            top, left = _random_crop_region(padded_h, padded_w, target_h, target_w)
            cropped = padded[top:top + target_h, left:left + target_w]
        elif data.shape[0] <= 4:
            # (C, H, W)
            padded_h, padded_w = padded.shape[1], padded.shape[2]
            top, left = _random_crop_region(padded_h, padded_w, target_h, target_w)
            cropped = padded[:, top:top + target_h, left:left + target_w]
        else:
            # (H, W, C)
            padded_h, padded_w = padded.shape[0], padded.shape[1]
            top, left = _random_crop_region(padded_h, padded_w, target_h, target_w)
            cropped = padded[top:top + target_h, left:left + target_w, :]

        return Tensor(cropped) if is_tensor else cropped
    

#| export
class Compose:
    """
    Compose multiple transforms into a pipeline.

    Applies transforms in sequence, passing output of each
    as input to the next.

    Args:
        transforms: List of transform callables
    """

    def __init__(self, transforms):
        """
        Initialize Compose with list of transforms.

        EXAMPLE:
        >>> transforms = Compose([
        ...     RandomHorizontalFlip(0.5),
        ...     RandomCrop(32, padding=4)
        ... ])
        """

        self.transforms = transforms

    def __call__(self, x):
        """Apply all transforms in sequence."""
        for transform in self.transforms:
            x = transform(x)
        return x

In [50]:


def test_unit_augmentation():
    """🧪 Test data augmentation transforms."""
    print("🧪 Unit Test: Data Augmentation...")

    # Test 1: RandomHorizontalFlip
    print("  Testing RandomHorizontalFlip...")
    flip = RandomHorizontalFlip(p=1.0)  # Always flip for deterministic test

    img = np.array([[1, 2, 3], [4, 5, 6]])  # 2x3 image
    flipped = flip(img)
    expected = np.array([[3, 2, 1], [6, 5, 4]])
    assert np.array_equal(flipped, expected), f"Flip failed: {flipped} vs {expected}"

    # Test never flip
    no_flip = RandomHorizontalFlip(p=0.0)
    unchanged = no_flip(img)
    assert np.array_equal(unchanged, img), "p=0 should never flip"

    # Test 2: RandomCrop shape preservation
    print("  Testing RandomCrop...")
    crop = RandomCrop(32, padding=4)

    # Test with (C, H, W) format (CIFAR-10 style)
    img_chw = np.random.randn(3, 32, 32)
    cropped = crop(img_chw)
    assert cropped.shape == (3, 32, 32), f"CHW crop shape wrong: {cropped.shape}"

    # Test with (H, W) format
    img_hw = np.random.randn(28, 28)
    crop_hw = RandomCrop(28, padding=4)
    cropped_hw = crop_hw(img_hw)
    assert cropped_hw.shape == (28, 28), f"HW crop shape wrong: {cropped_hw.shape}"

    # Test 3: Compose pipeline
    print("  Testing Compose...")
    transforms = Compose([
        RandomHorizontalFlip(p=0.5),
        RandomCrop(32, padding=4)
    ])

    img = np.random.randn(3, 32, 32)
    augmented = transforms(img)
    assert augmented.shape == (3, 32, 32), f"Compose output shape wrong: {augmented.shape}"

    # Test 4: Transforms work with Tensor
    print("  Testing Tensor compatibility...")
    tensor_img = Tensor(np.random.randn(3, 32, 32))

    flip_result = RandomHorizontalFlip(p=1.0)(tensor_img)
    assert isinstance(flip_result, Tensor), "Flip should return Tensor when given Tensor"

    crop_result = RandomCrop(32, padding=4)(tensor_img)
    assert isinstance(crop_result, Tensor), "Crop should return Tensor when given Tensor"

    # Test 5: Randomness verification
    print("  Testing randomness...")
    flip_random = RandomHorizontalFlip(p=0.5)

    # Run many times and check we get both outcomes
    flips = 0
    no_flips = 0
    test_img = np.array([[1, 2]])

    for _ in range(100):
        result = flip_random(test_img)
        if np.array_equal(result, np.array([[2, 1]])):
            flips += 1
        else:
            no_flips += 1

    # With p=0.5, we should get roughly 50/50 (allow for randomness)
    assert flips > 20 and no_flips > 20, f"Flip randomness seems broken: {flips} flips, {no_flips} no-flips"

    print("✅ Data Augmentation works correctly!")

if __name__ == "__main__":
    test_unit_augmentation()

🧪 Unit Test: Data Augmentation...
  Testing RandomHorizontalFlip...
  Testing RandomCrop...
  Testing Compose...
  Testing Tensor compatibility...
  Testing randomness...
✅ Data Augmentation works correctly!


In [51]:
def test_unit_dataloader():
    """🧪 Test DataLoader implementation."""
    print("🧪 Unit Test: DataLoader...")

    # Create test dataset
    features = Tensor([[1, 2], [3, 4], [5, 6], [7, 8], [9, 10]])  # 5 samples
    labels = Tensor([0, 1, 0, 1, 0])
    dataset = TensorDataset(features, labels)

    # Test basic batching (no shuffle)
    loader = DataLoader(dataset, batch_size=2, shuffle=False)

    # Test length calculation
    assert len(loader) == 3, f"Expected 3 batches, got {len(loader)}"  # ceil(5/2) = 3

    batches = list(loader)
    assert len(batches) == 3, f"Expected 3 batches, got {len(batches)}"

    # Test first batch
    batch_features, batch_labels = batches[0]
    assert batch_features.data.shape == (2, 2), f"Wrong batch features shape: {batch_features.data.shape}"
    assert batch_labels.data.shape == (2,), f"Wrong batch labels shape: {batch_labels.data.shape}"

    # Test last batch (should have 1 sample)
    batch_features, batch_labels = batches[2]
    assert batch_features.data.shape == (1, 2), f"Wrong last batch features shape: {batch_features.data.shape}"
    assert batch_labels.data.shape == (1,), f"Wrong last batch labels shape: {batch_labels.data.shape}"

    # Test that data is preserved
    assert np.array_equal(batches[0][0].data[0], [1, 2]), "First sample should be [1,2]"
    assert batches[0][1].data[0] == 0, "First label should be 0"

    # Test shuffling produces different order
    loader_shuffle = DataLoader(dataset, batch_size=5, shuffle=True)
    loader_no_shuffle = DataLoader(dataset, batch_size=5, shuffle=False)

    batch_shuffle = list(loader_shuffle)[0]
    batch_no_shuffle = list(loader_no_shuffle)[0]

    # Note: This might occasionally fail due to random chance, but very unlikely
    # We'll just test that both contain all the original data
    shuffle_features = set(tuple(row) for row in batch_shuffle[0].data)
    no_shuffle_features = set(tuple(row) for row in batch_no_shuffle[0].data)
    expected_features = {(1, 2), (3, 4), (5, 6), (7, 8), (9, 10)}

    assert shuffle_features == expected_features, "Shuffle should preserve all data"
    assert no_shuffle_features == expected_features, "No shuffle should preserve all data"

    print("✅ DataLoader works correctly!")

if __name__ == "__main__":
    test_unit_dataloader()

🧪 Unit Test: DataLoader...
✅ DataLoader works correctly!


In [52]:
def test_unit_dataloader_deterministic():
    """🧪 Test DataLoader deterministic shuffling with fixed seed."""
    print("🧪 Unit Test: DataLoader Deterministic Shuffling...")

    # Create test dataset
    features = Tensor([[1, 2], [3, 4], [5, 6], [7, 8]])
    labels = Tensor([0, 1, 0, 1])
    dataset = TensorDataset(features, labels)

    # Test that same seed produces same shuffle
    random.seed(42)
    loader1 = DataLoader(dataset, batch_size=2, shuffle=True)
    batches1 = list(loader1)

    random.seed(42)
    loader2 = DataLoader(dataset, batch_size=2, shuffle=True)
    batches2 = list(loader2)

    # Should produce identical batches with same seed
    for i, (batch1, batch2) in enumerate(zip(batches1, batches2)):
        assert np.array_equal(batch1[0].data, batch2[0].data), \
            f"Batch {i} features should be identical with same seed"
        assert np.array_equal(batch1[1].data, batch2[1].data), \
            f"Batch {i} labels should be identical with same seed"

    # Test that different seeds produce different shuffles
    random.seed(42)
    loader3 = DataLoader(dataset, batch_size=2, shuffle=True)
    batches3 = list(loader3)

    random.seed(123)  # Different seed
    loader4 = DataLoader(dataset, batch_size=2, shuffle=True)
    batches4 = list(loader4)

    # Should produce different batches with different seeds (very likely)
    different = False
    for batch3, batch4 in zip(batches3, batches4):
        if not np.array_equal(batch3[0].data, batch4[0].data):
            different = True
            break

    assert different, "Different seeds should produce different shuffles"

    print("✅ Deterministic shuffling works correctly!")

if __name__ == "__main__":
    test_unit_dataloader_deterministic()

🧪 Unit Test: DataLoader Deterministic Shuffling...
✅ Deterministic shuffling works correctly!


In [54]:
def analyze_dataloader_performance():
    """📊 Analyze DataLoader performance characteristics."""
    print("📊 Analyzing DataLoader Performance...")

    # Create test dataset of varying sizes
    sizes = [1000, 5000, 10000]
    batch_sizes = [16, 64, 256]

    print("\n🔍 Batch Size vs Loading Time:")

    for size in sizes:
        # Create synthetic dataset
        features = Tensor(np.random.randn(size, 100))  # 100 features
        labels = Tensor(np.random.randint(0, 10, size))
        dataset = TensorDataset(features, labels)

        print(f"\nDataset size: {size} samples")

        for batch_size in batch_sizes:
            # Time data loading
            loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

            start_time = time.time()
            batch_count = 0
            for batch in loader:
                batch_count += 1
            end_time = time.time()

            elapsed = end_time - start_time
            throughput = size / elapsed if elapsed > 0 else float('inf')

            print(f"  Batch size {batch_size:3d}: {elapsed:.3f}s ({throughput:,.0f} samples/sec)")

    # Analyze shuffle overhead
    print("\n🔄 Shuffle Overhead Analysis:")

    dataset_size = 10000
    features = Tensor(np.random.randn(dataset_size, 50))
    labels = Tensor(np.random.randint(0, 5, dataset_size))
    dataset = TensorDataset(features, labels)

    batch_size = 64

    # No shuffle
    loader_no_shuffle = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    start_time = time.time()
    batches_no_shuffle = list(loader_no_shuffle)
    time_no_shuffle = time.time() - start_time

    # With shuffle
    loader_shuffle = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    start_time = time.time()
    batches_shuffle = list(loader_shuffle)
    time_shuffle = time.time() - start_time

    shuffle_overhead = ((time_shuffle - time_no_shuffle) / time_no_shuffle) * 100

    print(f"  No shuffle: {time_no_shuffle:.3f}s")
    print(f"  With shuffle: {time_shuffle:.3f}s")
    print(f"  Shuffle overhead: {shuffle_overhead:.1f}%")

    print("\n💡 Key Insights:")
    print("• Larger batch sizes reduce per-sample overhead")
    print("• Shuffle adds minimal overhead for reasonable dataset sizes")
    print("• Memory usage scales linearly with batch size")
    print("🚀 Production tip: Balance batch size with GPU memory limits")


def analyze_memory_usage():
    """📊 Analyze memory usage patterns in data loading."""
    print("\n📊 Analyzing Memory Usage Patterns...")

    # Memory usage estimation
    def estimate_memory_mb(batch_size, feature_size, dtype_bytes=4):
        """Estimate memory usage for a batch."""
        return (batch_size * feature_size * dtype_bytes) / (1024 * 1024)

    print("\n💾 Memory Usage by Batch Configuration:")

    feature_sizes = [784, 3072, 50176]  # MNIST, CIFAR-10, ImageNet-like
    feature_names = ["MNIST (28×28)", "CIFAR-10 (32×32×3)", "ImageNet (224×224×1)"]
    batch_sizes = [1, 32, 128, 512]

    for feature_size, name in zip(feature_sizes, feature_names):
        print(f"\n{name}:")
        for batch_size in batch_sizes:
            memory_mb = estimate_memory_mb(batch_size, feature_size)
            print(f"  Batch {batch_size:3d}: {memory_mb:6.1f} MB")

    print("\n🎯 Memory Trade-offs:")
    print("• Larger batches: More memory, better GPU utilization")
    print("• Smaller batches: Less memory, more noisy gradients")
    print("• Sweet spot: Usually 32-128 depending on model size")

    # Demonstrate actual memory usage with our tensors
    print("\n🧪 Actual Tensor Memory Usage:")

    # Create different sized tensors
    tensor_small = Tensor(np.random.randn(32, 784))    # Small batch
    tensor_large = Tensor(np.random.randn(512, 784))   # Large batch

    # Measure actual memory (data array + object overhead)
    small_bytes = tensor_small.data.nbytes
    large_bytes = tensor_large.data.nbytes

    # Also measure Python object overhead
    small_total = sys.getsizeof(tensor_small.data) + sys.getsizeof(tensor_small)
    large_total = sys.getsizeof(tensor_large.data) + sys.getsizeof(tensor_large)

    print("  Small batch (32×784):")
    print(f"    - Data only: {small_bytes / 1024:.1f} KB")
    print(f"    - With object overhead: {small_total / 1024:.1f} KB")
    print("  Large batch (512×784):")
    print(f"    - Data only: {large_bytes / 1024:.1f} KB")
    print(f"    - With object overhead: {large_total / 1024:.1f} KB")
    print(f"  Ratio: {large_bytes / small_bytes:.1f}× (data scales linearly)")

    print("\n🎯 Memory Optimization Tips:")
    print("• Object overhead becomes negligible with larger batches")
    print("• Use float32 instead of float64 to halve memory usage")
    print("• Consider gradient accumulation for effective larger batches")


def analyze_collation_overhead():
    """📊 Analyze the cost of collating samples into batches."""
    print("\n📊 Analyzing Collation Overhead...")

    # Test different batch sizes to see collation cost
    dataset_size = 1000
    feature_size = 100
    features = Tensor(np.random.randn(dataset_size, feature_size))
    labels = Tensor(np.random.randint(0, 10, dataset_size))
    dataset = TensorDataset(features, labels)

    print("\n⚡ Collation Time by Batch Size:")

    for batch_size in [8, 32, 128, 512]:
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

        start_time = time.time()
        for batch in loader:
            pass  # Just iterate, measuring collation overhead
        total_time = time.time() - start_time

        batches = len(loader)
        time_per_batch = (total_time / batches) * 1000  # Convert to ms

        print(f"  Batch size {batch_size:3d}: {time_per_batch:.2f}ms per batch ({batches} batches total)")

    print("\n💡 Collation Insights:")
    print("• Larger batches take longer to collate (more np.stack operations)")
    print("• But fewer large batches are more efficient than many small ones")
    print("• Optimal: Balance between batch size and iteration overhead")


# Run the systems analysis (uncomment to run)
if __name__ == "__main__":
    analyze_dataloader_performance()
    analyze_memory_usage()
    analyze_collation_overhead()

📊 Analyzing DataLoader Performance...

🔍 Batch Size vs Loading Time:

Dataset size: 1000 samples
  Batch size  16: 0.005s (209,119 samples/sec)
  Batch size  64: 0.006s (166,606 samples/sec)
  Batch size 256: 0.008s (133,017 samples/sec)

Dataset size: 5000 samples
  Batch size  16: 0.017s (301,202 samples/sec)
  Batch size  64: 0.013s (393,543 samples/sec)
  Batch size 256: 0.014s (345,831 samples/sec)

Dataset size: 10000 samples
  Batch size  16: 0.023s (430,185 samples/sec)
  Batch size  64: 0.024s (408,913 samples/sec)
  Batch size 256: 0.026s (391,530 samples/sec)

🔄 Shuffle Overhead Analysis:
  No shuffle: 0.022s
  With shuffle: 0.027s
  Shuffle overhead: 23.1%

💡 Key Insights:
• Larger batch sizes reduce per-sample overhead
• Shuffle adds minimal overhead for reasonable dataset sizes
• Memory usage scales linearly with batch size
🚀 Production tip: Balance batch size with GPU memory limits

📊 Analyzing Memory Usage Patterns...

💾 Memory Usage by Batch Configuration:

MNIST (28×2